In [ ]:

!unzip "/content/drive/MyDrive/helmet detection.v2i.yolov11.zip" -d /content/dataset


Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/train/labels/003798_jpg.rf.a6e0777c15bb7fe6672aeeea939d94f3.txt  
  inflating: /content/dataset/train/labels/004872_jpg.rf.ea14734c6f9f9ef311431f931f9bce69.txt  
  inflating: /content/dataset/train/labels/002977_jpg.rf.6c4f599ceae9471927b38ab62c411fdf.txt  
  inflating: /content/dataset/train/labels/001515_jpg.rf.d755275d6c3b911f135ad17489d30e7f.txt  
  inflating: /content/dataset/train/labels/000162_jpg.rf.fd638b02e96de8b36c3dfe95486d459b.txt  
  inflating: /content/dataset/train/labels/002636_jpg.rf.fd49642bd97111c7d63ed011beeeb2d0.txt  
  inflating: /content/dataset/train/labels/000600_jpg.rf.f03314bb16f740814332d0896eaeb0b4.txt  
  inflating: /content/dataset/train/labels/001890_jpg.rf.a2a73e771dabfec4d885165328dd9b2e.txt  
  inflating: /content/dataset/train/labels/001118_jpg.rf.d5d57f441f11abc09f92bfaead9fa384.txt  
  inflating: /content/dataset/train/labels/ppe_0727_jpg.rf.fb88de0a98ecabee48f28a19f1

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
os.listdir("/content/dataset")

Harusnya muncul:

train/
valid/
test/
data.yaml


SyntaxError: invalid syntax (2502043424.py, line 4)

In [ ]:
import os

dataset_path = None

for root, dirs, files in os.walk("/content/dataset"):
    if "data.yaml" in files:
        dataset_path = root
        break

if dataset_path is None:
    raise FileNotFoundError("data.yaml tidak ditemukan. Cek lagi file zip dataset.")

print("Dataset path:", dataset_path)
print("Isi folder dataset:", os.listdir(dataset_path))

Dataset path: /content/dataset
Isi folder dataset: ['data.yaml', 'README.dataset.txt', 'valid', 'train', 'README.roboflow.txt', 'test']


In [ ]:
# ============================================================
# 5. Baca data.yaml
# ============================================================
import yaml
import os

yaml_path = os.path.join(dataset_path, "data.yaml")

with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

print("Data YAML awal:")
print(data_yaml)

Data YAML awal:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['helmet', 'no_helmet', 'person'], 'roboflow': {'workspace': '039s-workspace', 'project': 'helmet-detection-wagvw', 'version': 2, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/039s-workspace/helmet-detection-wagvw/dataset/2'}}


In [ ]:
# ============================================================
# 6. Fix Path data.yaml untuk Colab
# ============================================================
data_yaml["train"] = os.path.join(dataset_path, "train/images")
data_yaml["val"] = os.path.join(dataset_path, "valid/images")
data_yaml["test"] = os.path.join(dataset_path, "test/images")

# Rapikan format names
if "names" in data_yaml:
    names = data_yaml["names"]
    if isinstance(names, dict):
        names = [names[i] for i in range(len(names))]
    data_yaml["names"] = names
    data_yaml["nc"] = len(names)

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Data YAML setelah diperbaiki:")
print(data_yaml)

Data YAML setelah diperbaiki:
{'train': '/content/dataset/train/images', 'val': '/content/dataset/valid/images', 'test': '/content/dataset/test/images', 'nc': 3, 'names': ['helmet', 'no_helmet', 'person'], 'roboflow': {'workspace': '039s-workspace', 'project': 'helmet-detection-wagvw', 'version': 2, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/039s-workspace/helmet-detection-wagvw/dataset/2'}}


In [ ]:
# ============================================================
# 8. Cleaning Dataset
# Hapus label kosong, gambar tanpa label, dan label tanpa gambar
# ============================================================
from pathlib import Path

IMG_EXTS = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

def clean_dataset(split):
    img_dir = Path(dataset_path) / split / "images"
    label_dir = Path(dataset_path) / split / "labels"

    print(f"\nMembersihkan split: {split}")

    empty_count = 0
    for label_path in label_dir.glob("*.txt"):
        if label_path.stat().st_size == 0:
            label_path.unlink()
            empty_count += 1

    removed_images = 0
    for img_path in img_dir.iterdir():
        if img_path.suffix in IMG_EXTS:
            label_path = label_dir / (img_path.stem + ".txt")
            if not label_path.exists():
                img_path.unlink()
                removed_images += 1

    image_stems = {p.stem for p in img_dir.iterdir() if p.suffix in IMG_EXTS}
    removed_labels = 0

    for label_path in label_dir.glob("*.txt"):
        if label_path.stem not in image_stems:
            label_path.unlink()
            removed_labels += 1

    print("Label kosong dihapus:", empty_count)
    print("Gambar tanpa label dihapus:", removed_images)
    print("Label tanpa gambar dihapus:", removed_labels)

for split in ["train", "valid", "test"]:
    clean_dataset(split)


Membersihkan split: train
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0

Membersihkan split: valid
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0

Membersihkan split: test
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0


In [ ]:
# ============================================================
# 9. Validasi Format Label YOLO
# ============================================================
from pathlib import Path

NC = data_yaml["nc"]
print("Jumlah kelas:", NC)
print("Nama kelas:", data_yaml["names"])

def validate_labels(split):
    label_dir = Path(dataset_path) / split / "labels"
    bad_files = []

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line_number, line in enumerate(lines, start=1):
            parts = line.strip().split()

            if len(parts) != 5:
                bad_files.append((label_path.name, line_number, "jumlah kolom bukan 5", line.strip()))
                break

            try:
                cls = int(float(parts[0]))
                values = list(map(float, parts[1:]))
            except:
                bad_files.append((label_path.name, line_number, "format angka salah", line.strip()))
                break

            if cls < 0 or cls >= NC:
                bad_files.append((label_path.name, line_number, f"class id salah: {cls}", line.strip()))
                break

            if any(v < 0 or v > 1 for v in values):
                bad_files.append((label_path.name, line_number, "koordinat di luar 0-1", line.strip()))
                break

    print(f"{split} label bermasalah:", len(bad_files))

    for item in bad_files[:10]:
        print(item)

    return bad_files

all_bad = []

for split in ["train", "valid", "test"]:
    all_bad.extend(validate_labels(split))

print("Total semua label bermasalah:", len(all_bad))

Jumlah kelas: 3
Nama kelas: ['helmet', 'no_helmet', 'person']
train label bermasalah: 0
valid label bermasalah: 0
test label bermasalah: 1
('005377_jpg.rf.877294944cd6d730dd900feb00280ffe.txt', 2, 'jumlah kolom bukan 5', '0 0.5925 0.975 0.5900000000000001 0.9016666666666666 0.5775 0.8783333333333333 0.5575 0.7183333333333334 0.60375 0.7 0.6375 0.8816666666666666 0.675 0.8716666666666667 0.6625 0.7016666666666667 0.665 0.6716666666666666 0.7025 0.5916666666666667 0.7125 0.3983333333333333 0.7025 0.335 0.6625 0.2783333333333333 0.65 0.23166666666666663 0.6525000000000001 0.19833333333333333 0.6162500000000001 0.16 0.58875 0.16666666666666669 0.5650000000000001 0.19166666666666668 0.5625 0.24833333333333335 0.5650000000000001 0.2783333333333333 0.5825 0.29166666666666663 0.60375 0.33666666666666667 0.6225 0.3416666666666667 0.62 0.3783333333333333 0.60875 0.4066666666666666 0.5537500000000001 0.4 0.535 0.4116666666666666 0.5375 0.43166666666666664 0.5925 0.505 0.5900000000000001 0.5916666

In [ ]:
import os

def remove_empty_labels(label_dir):
    for file in os.listdir(label_dir):
        path = os.path.join(label_dir, file)
        if os.path.getsize(path) == 0:
            print("Hapus label kosong:", file)
            os.remove(path)

remove_empty_labels("/content/dataset/train/labels")
remove_empty_labels("/content/dataset/valid/labels")
remove_empty_labels("/content/dataset/test/labels")
def remove_images_without_labels(img_dir, label_dir):
    for img in os.listdir(img_dir):
        label = img.replace(".jpg", ".txt")
        if not os.path.exists(os.path.join(label_dir, label)):
            print("Hapus gambar tanpa label:", img)
            os.remove(os.path.join(img_dir, img))

remove_images_without_labels("/content/dataset/train/images", "/content/dataset/train/labels")
remove_images_without_labels("/content/dataset/valid/images", "/content/dataset/valid/labels")
remove_images_without_labels("/content/dataset/test/images", "/content/dataset/test/labels")


In [ ]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  # WAJIB jalan dulu, baru lanjut ke training


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# ============================================================
# 10. Training YOLOv11s Helmet Detection
# ============================================================
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/helmet_detection_yolo",
    name="yolo11s_helmet_detection_img640",
    workers=2,
    patience=30,
    save=True,
    plots=True,
    cache=False,
    cos_lr=True,
    close_mosaic=10
)

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_helmet_detection_img640, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [ ]:
# ============================================================
# 11. Evaluasi Best Model pada Test Set
# ============================================================
from ultralytics import YOLO
import os

best_model_path = "/content/helmet_detection_yolo/yolo11s_helmet_detection_img640/weights/best.pt"

if not os.path.exists(best_model_path):
    raise FileNotFoundError("best.pt tidak ditemukan. Pastikan training sudah selesai.")

model = YOLO(best_model_path)

metrics = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project="/content/helmet_detection_eval",
    name="final_test_yolo11s_img640",
    exist_ok=True
)

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
f1_score = 2 * (precision * recall) / (precision + recall + 1e-9)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

print("Precision :", precision)
print("Recall    :", recall)
print("F1-score  :", f1_score)
print("mAP@50    :", map50)
print("mAP@50:95 :", map50_95)

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 21.2±12.8 MB/s, size: 60.7 KB)
val: Scanning /content/dataset/test/labels... 869 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 869/869 541.2it/s 1.6s
val: New cache created: /content/dataset/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 3, len(boxes) = 3271. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 7.4it/s 14.7s
                   all        869       3271      0.843      0.809       0.85      0.563
                helmet        716       2110      0.905      0.914      0.947 

In [ ]:
# ============================================================
# 12. Hitung Inference Time dan FPS
# ============================================================
import glob
import os
import time
import torch

TEST_IMG_DIR = os.path.join(dataset_path, "test/images")

test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(glob.glob(os.path.join(TEST_IMG_DIR, ext)))

print("Jumlah gambar test:", len(test_images))

# Warm up
for img_path in test_images[:10]:
    _ = model.predict(img_path, imgsz=640, conf=0.25, verbose=False)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.time()

for img_path in test_images:
    _ = model.predict(img_path, imgsz=640, conf=0.25, verbose=False)

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_time = time.time()

total_inference_time = end_time - start_time
avg_inference_time = total_inference_time / len(test_images)
fps = 1 / avg_inference_time

print("Total inference time:", total_inference_time, "detik")
print("Average inference time:", avg_inference_time, "detik/gambar")
print("FPS:", fps)

Jumlah gambar test: 869
Total inference time: 12.592427015304565 detik
Average inference time: 0.014490710029119178 detik/gambar
FPS: 69.00973092350156


In [ ]:
# ============================================================
# 13. Simpan Metrik ke CSV
# ============================================================
import pandas as pd
import os

OUTPUT_DIR = "/content/helmet_detection_eval/yolo11s_final_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

summary = pd.DataFrame([{
    "Model": "YOLOv11s",
    "Dataset": "Helmet Detection",
    "Jumlah Kelas": data_yaml["nc"],
    "Kelas": ", ".join(data_yaml["names"]),
    "Epochs": 100,
    "Image Size": 640,
    "Batch": 16,
    "Accuracy": "",
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1_score,
    "mAP@50": map50,
    "mAP@50:95": map50_95,
    "Inference Time per Image (s)": avg_inference_time,
    "FPS": fps
}])

summary_path = os.path.join(OUTPUT_DIR, "yolo11s_final_metrics.csv")
summary.to_csv(summary_path, index=False)

summary

,Model,Dataset,Jumlah Kelas,Kelas,Epochs,Image Size,Batch,Accuracy,Precision,Recall,F1-Score,mAP@50,mAP@50:95,Inference Time per Image (s),FPS
0,YOLOv11s,Helmet Detection,3,"helmet, no_helmet, person",100,640,16,,0.843056,0.809399,0.825885,0.849647,0.562563,0.014491,69.009731


In [ ]:
# ============================================================
# 14. Simpan Sample Prediction
# ============================================================
model.predict(
    source=TEST_IMG_DIR,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/helmet_detection_eval",
    name="yolo11s_prediction_sample",
    exist_ok=True
)

Results saved to /content/helmet_detection_eval/yolo11s_prediction_sample


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'helmet', 1: 'no_helmet', 2: 'person'}
 obb: None
 orig_img: array([[[ 98, 117,  98],
         [102, 121, 102],
         [105, 125, 106],
         ...,
         [ 76,  94,  87],
         [ 74,  92,  85],
         [ 73,  91,  84]],
 
        [[ 99, 118,  99],
         [103, 122, 103],
         [106, 126, 107],
         ...,
         [ 76,  94,  87],
         [ 74,  92,  85],
         [ 73,  91,  84]],
 
        [[101, 120, 101],
         [105, 124, 105],
         [107, 127, 108],
         ...,
         [ 75,  93,  86],
         [ 73,  91,  84],
         [ 73,  91,  84]],
 
        ...,
 
        [[220, 236, 212],
         [221, 237, 213],
         [222, 238, 214],
         ...,
         [ 95, 129, 129],
         [ 83, 117, 117],
         [ 87, 121, 121]],
 
        [[220, 236, 212],
         [221, 237, 213],
         [222, 238, 214],
   

In [ ]:
# ============================================================
# 15. Kumpulkan Semua Hasil ke Folder Final
# ============================================================
import os
import shutil
import glob

RESULT_FOLDER = "/content/YOLOv11s_Helmet_Detection_Final_Result"

if os.path.exists(RESULT_FOLDER):
    shutil.rmtree(RESULT_FOLDER)

os.makedirs(RESULT_FOLDER, exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/01_Model", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/02_Training_Result", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/03_Evaluation_Result", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/04_Visualization", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/05_Prediction_Sample", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/06_Config", exist_ok=True)

TRAIN_DIR = "/content/helmet_detection_yolo/yolo11s_helmet_detection_img640"
EVAL_DIR = "/content/helmet_detection_eval/final_test_yolo11s_img640"
PRED_DIR = "/content/helmet_detection_eval/yolo11s_prediction_sample"

# Copy model
if os.path.exists(f"{TRAIN_DIR}/weights/best.pt"):
    shutil.copy(f"{TRAIN_DIR}/weights/best.pt", f"{RESULT_FOLDER}/01_Model/best_yolov11s.pt")

if os.path.exists(f"{TRAIN_DIR}/weights/last.pt"):
    shutil.copy(f"{TRAIN_DIR}/weights/last.pt", f"{RESULT_FOLDER}/01_Model/last_yolov11s.pt")

# Copy hasil training
for file in glob.glob(f"{TRAIN_DIR}/*"):
    if os.path.isfile(file):
        shutil.copy(file, f"{RESULT_FOLDER}/02_Training_Result/{os.path.basename(file)}")

# Copy hasil evaluasi CSV
if os.path.exists(OUTPUT_DIR):
    for file in glob.glob(f"{OUTPUT_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/03_Evaluation_Result/{os.path.basename(file)}")

# Copy hasil evaluasi test
if os.path.exists(EVAL_DIR):
    for file in glob.glob(f"{EVAL_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/03_Evaluation_Result/{os.path.basename(file)}")

# Copy visualisasi training
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    for file in glob.glob(f"{TRAIN_DIR}/{ext}"):
        shutil.copy(file, f"{RESULT_FOLDER}/04_Visualization/{os.path.basename(file)}")

# Copy prediction sample
if os.path.exists(PRED_DIR):
    for file in glob.glob(f"{PRED_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/05_Prediction_Sample/{os.path.basename(file)}")

# Copy config
shutil.copy(yaml_path, f"{RESULT_FOLDER}/06_Config/data.yaml")

# README
readme = f"""
YOLOv11s Helmet Detection Final Result

Judul:
Deteksi Kepatuhan Penggunaan Helm Keselamatan Menggunakan Perbandingan YOLOv5, YOLOv8, dan YOLOv11 pada Dataset Publik

Model:
YOLOv11s

Dataset:
Helmet Detection

Class:
{", ".join(data_yaml["names"])}

Setting Training:
- Epochs: 100
- Image Size: 640
- Batch: 16
- Optimizer: default Ultralytics
- patience: 30
- cos_lr: True
- close_mosaic: 10
- cache: False

Metrik:
- Precision
- Recall
- F1-score
- mAP@50
- mAP@50:95
- Inference Time
- FPS
- Confusion Matrix
"""

with open(f"{RESULT_FOLDER}/README.txt", "w") as f:
    f.write(readme)

print("Folder hasil dibuat:", RESULT_FOLDER)

Folder hasil dibuat: /content/YOLOv11s_Helmet_Detection_Final_Result


In [ ]:
# ============================================================
# 16. Simpan Folder Final ke Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TARGET = "/content/drive/MyDrive/UAS_AI_Helmet_Detection/YOLOv11s_Final_Result"

if os.path.exists(DRIVE_TARGET):
    shutil.rmtree(DRIVE_TARGET)

shutil.copytree(RESULT_FOLDER, DRIVE_TARGET)

print("Hasil YOLOv11s Helmet Detection sudah masuk ke Google Drive:")
print(DRIVE_TARGET)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hasil YOLOv11s Helmet Detection sudah masuk ke Google Drive:
/content/drive/MyDrive/UAS_AI_Helmet_Detection/YOLOv11s_Final_Result
